# Provider Alignment — Toy Replication

A runnable, self-contained walkthrough of the NScreen provider-alignment pipeline (Phases 4–6). Every step below is the real production SQL, adapted to Trino and a tiny hand-built dataset in `iceberg.jteixeira_ipa`, so you can watch each transformation happen on data small enough to check by eye.

You do not need to read anything else first. The companion [`provider-alignment-phase.md`](docs/analysis/provider-alignment-phase.md) covers the same ground with production-scale measurements and a full edge-case taxonomy; [`provider-alignment-defects.md`](docs/analysis/provider-alignment-defects.md) documents two defects found while writing it.

## The problem being solved

Three data vendors — **LiveRamp**, **Throtle** and **Experian** — each tell us which cookies belong to the same person. They do not coordinate. Each assigns its own opaque group id, and the same real person may be group `T1` to Throtle and group `L1` to LiveRamp.

| Term | Meaning |
|---|---|
| `uid` | A cookie / device identifier. The thing we observe in the wild. |
| `matchid` | A vendor's *person-group* id. Several UIDs sharing a matchid = the vendor's claim that they are one person. |
| `reason` | A provenance code recording *how* a row's matchid was decided (`LR`, `TH`, `LRTH`, …). |

The job of provider alignment is to **merge these three id namespaces into one**, using the UIDs that appear in more than one vendor's feed as translation evidence. If Throtle's `T1` and LiveRamp's `L1` share enough cookies, they are the same person, and we relabel `T1`'s members with `L1`.

Precedence is fixed: **LiveRamp matchids always win.** An accepted merge rewrites the Throtle side (`coalesce(m.lrmatchid, t.matchid)`); a LiveRamp matchid is never rewritten, never dropped, and never merged into another LiveRamp matchid anywhere in the pipeline.

## The shape of the pipeline

Two SQL jobs run in sequence and write into one table with a `stage` column, then three later phases append more stages to the same table:

| Phase | What it does | Stage written |
|---|---|---|
| Phase 4 Job 1 | Align LiveRamp ↔ Throtle | `lrth` (production) / `initial` + `lrth` (this notebook — see warning) |
| Phase 4 Job 2 | Align the result ↔ Experian | `initial` |
| Phase 5a | Copy graph-connected rows as propagation seeds | `connected_ns` |
| Phase 5b | Spread matchids along IP-collocation edges, twice | `stage1`, `stage2` |
| Phase 6 | Cluster whatever is left with Louvain | `louvain` |

The final result table is a pure filter: `stage in ('initial','stage1','stage2','louvain')`. `lrth` and `connected_ns` are intermediate scratch.

> ### ⚠️ Job 1's `stage` labels are a lie — and this notebook reproduces the truth
>
> Job 1's SQL ends in a union labelling `lr_out` as `stage = 'initial'` and `th_out` as `stage = 'lrth'`, so it reads as though it writes two stages. **It does not.** [`common.py:49-51`](src/nscreen_graph/spark/common.py#L49-L51) overwrites every partition column with the literal the Luigi task supplies:
>
> ```python
> for col_name, col_value in args.partition_columns:
>     df = df.withColumn(col_name, F.lit(col_value))
> ```
>
> That literal comes from each task's `args()` in [`daily.py`](src/nscreen_graph/tasks/daily.py). Job 1 supplies `("stage", "lrth")` ([`daily.py:297-300`](src/nscreen_graph/tasks/daily.py#L297-L300)):
>
> ```python
> def args(self) -> Args:
>     return Args(
>         sql_file_name="NScreenLiverampThrotleOnlyStage.sql",
>         destination_table="iceberg.crossscreen.nscreen_liveramp_throtle_ex_reason_staged",
>         partition_columns=[("day", self.day), ("stage", "lrth")],
>     )
> ```
>
> so *all* of Job 1's output, LiveRamp-rooted rows included, lands in `lrth` regardless of which branch (`lr_out` or `th_out`) a row came from. Job 2's task supplies `("stage", "initial")` the same way ([`daily.py:326-328`](src/nscreen_graph/tasks/daily.py#L326-L328)) and is the only writer of `initial`.
>
> **This notebook replicates that real behaviour, not the SQL's stated intent.** Two consequences you will see below and would not see from reading the SQL:
>
> - Job 2 Experian-checks **everything**, not just Throtle leftovers — which is where the reason `LREX` comes from (~1.6B rows/day in production). The `L6`/`E3` group in the toy data exists to demonstrate it.
> - Experian merges apply at **matchid** level, so `v1`, `v2` and `v9` end up `LRTHEX` despite having no Experian row of their own — they simply belong to a group that earned the merge via `v3`. This is the same mechanism as the real production UID `GvT54vuou0s4`.
>
> Full analysis and proofs: [`provider-alignment-defects.md`](docs/analysis/provider-alignment-defects.md).

## Adaptations from the real SQL

| Step | Real source file | Adaptations |
|---|---|---|
| Job 1 | `NScreenLiverampThrotleOnlyStage.sql` | no `day` filter, no `householdid`/`stable` columns |
| Job 2 | `NScreenLiverampThrotleExInitialStage.sql` | same; reads `nscreen_toy_lrth_all` (= Job 1's full output) instead of `staged where stage='lrth'` — identical contents |
| connected_ns | `NScreenLiverampThrotleExConnectedNsStage.sql` | `explode(array(...))` → `union` |
| stage1/stage2 | `NScreenLiverampThrotleExNStage.sql` | `inline(array(named_struct...))` → `union all`; no `stable` weight-doubling; stage2 seed filter `stage > 'stage0' and stage < 'stage2'` → explicit `in ('connected_ns','stage1')`; ⚠️ real `grouped` keys on `(uid, matchid, householdid, stable)` while toy keys on `(uid, matchid)` — in production a matchid seen under two householdids gets its weight **split** into separate candidates, not summed |
| remainder | `NScreenIpCollocationDailyExRemainder.sql` | toy geo table (all UIDs in one geo bucket) |
| louvain | `NScreenExRemainderLouvainStage.sql` | **SIMULATED** — the real step calls the Spark UDAF `hudf_calc_screen7_ids_udaf`; here the single surviving edge's endpoints both get `C1` |
| final result | `NScreenLrThrotleExReasonResult.sql` | `matchid <> 0` and `S_` stable-prefix skipped (varchar matchids, no stable column); junk-uid regexes skipped |

**Matchid typing matches production.** Matchids here are `bigint`, as they are in production (murmur3 hashes), because the type is load-bearing: `order by matchid desc` decides propagation tie-breaks and must compare numerically, not lexicographically.

Every matchid is written out literally in the cells below. The legend:

| Vendor | Matchids |
|---|---|
| Throtle | `T1`=101, `T2`=102, `T3`=103, `T4`=104, `T5`=105, `T6`=106, `T9`=109 |
| LiveRamp | `L1`=201, `L2`=202, `L3`=203, `L4`=204, `L5`=205, `L6`=206 |
| Experian | `E1`=301, `E2`=302, `E3`=303 |
| Louvain | `C1`=901 |

Values ascend with their labels, so every documented tie-break outcome still holds (`v20`→`T5`, the largest of `T3`/`T4`/`T5`). One honest difference remains: real hashes are frequently *negative*, so in production that tie-break is effectively arbitrary rather than "highest label wins".

All intermediates land in one `nscreen_toy_staged` table with a `stage` column — mirroring the real `nscreen_liveramp_throtle_ex_reason_staged` design — so the diagnostic `count(distinct ...)` query from the document runs against it unmodified.

⚠️ Running this notebook **recreates** every `nscreen_toy_*` table from scratch (idempotent top-to-bottom).

## How a merge is decided: mutual-best rank

Both Job 1 and Job 2 use the identical algorithm to decide whether two provider groups describe the same person. Understanding these five steps is enough to predict almost everything the pipeline does:

1. **Inner-join the two sources on `uid`.** Only cookies present in *both* feeds count as evidence.
2. **Group by the pair of matchids** and count shared UIDs per candidate pairing (`cnt`).
3. **`rank()` each pairing twice** — once within the first provider's matchid partition (by `cnt` descending), once within the second's. These are `rnk_1` and `rnk_2`.
4. **Keep only pairings ranked #1 in *both* directions** (`where rnk_1 = 1 and rnk_2 = 1`). This is the "mutual best" condition: each side must consider the other its strongest partner.
5. **Reject any matchid that still has more than one surviving pairing.** Checked on both sides, via `count1` and `count2`.

Step 5 is where most failed merges die, and it is worth dwelling on because `rank()` is used rather than `row_number()` — so equal counts *both* get rank 1 and both survive step 4.

```sql
limited as (
    select throtmatchid, lrmatchid, ...,
        count(*) over (partition by throtmatchid) as count1,
        count(*) over (partition by lrmatchid)    as count2
    from ranked
    where rnk_1 = 1 and rnk_2 = 1     -- the counters only ever see survivors of this
),
mapping as (
    select ... from limited where count1 = 1 and count2 = 1
)
```

| Counter | Partitioned by | Answers | `> 1` means |
|---|---|---|---|
| `count1` | `throtmatchid` | how many LiveRamp groups is this Throtle group still mutual-best with? | one Throtle group tied across LiveRamp groups |
| `count2` | `lrmatchid` | how many Throtle groups are still mutual-best with this LiveRamp group? | rival Throtle groups tied over one LiveRamp group |

`mapping` requires **both** to equal 1. So **ties are rejected, not broken** — when two pairings tie, *every* pairing involved is discarded and nothing merges at all. The `T6`/`L4`/`L5` case built into the toy data below exists to demonstrate exactly this.

Two traps worth internalising:

- **The counters count post-ranking survivors, not raw candidates.** A matchid can have many candidate pairings and still reach `count = 1`, because the weaker ones were eliminated in step 4. Having competition is not the same as having a tie.
- **The names are positional, so they mean different things per job.** Job 2 reuses this code partitioning by `exmatchid` (`count1`) and `lrthmatchid` (`count2`) — so there, `count1` is the *Experian* side.

## Reason codes

Every output row carries a `reason` recording how its matchid was decided. This is the provenance trail, and reading it is how you tell an evidence-backed merge from a fallback guess:

| Code | Meaning | Appears in this notebook? |
|---|---|---|
| `LR` | LiveRamp group with no Throtle mutual-best match | yes |
| `TH` | Throtle group with no LiveRamp mutual-best match | yes |
| `LRTH` | LiveRamp and Throtle mutually aligned | yes |
| `LRTHEX` | A merged LiveRamp+Throtle group that also aligned with Experian | yes |
| `THEX` | A Throtle-only group that also aligned with Experian | no — grammar allows it, but neither Experian group here lands on a Throtle-only group |
| `EX` | Experian-only, no alignment | yes |
| `LREX` | A LiveRamp group that aligned with Experian | yes — `v30`, `v31`, `v32` via the `L6`↔`E3` merge |
| `NA` | Placeholder written by the `connected_ns` copy stage; never reaches the final result | yes |
| `IC` | Assigned by IP-collocation propagation (`stage1`/`stage2`) | yes |
| `LU` | Assigned by Louvain clustering | yes |

Note the suffix grammar: Job 2 appends `EX` to whatever reason a row already had (`concat(lrth.reason, 'EX')`), which is why the codes read as a lineage rather than a category.

## Cast of UIDs

The toy dataset is designed so that every distinct behaviour of the pipeline shows up in at least one UID. Nothing here is filler:

| uid | Story |
| --- | --- |
| `v1` | Clean, direct LiveRamp member; its group merges with Throtle; also IP-graph connected |
| `v2` | Same merged group, but not graph-connected |
| `v4`, `v10` | Clean members of a second merged group |
| `v9` | LiveRamp member of `L1` with no Throtle row of its own — inherits the group's merge anyway; also graph-connected to `v24`/`v25`/`v26`, but a connection to already-resolved UIDs, not a new assignment |
| `v3` | Throtle-leftover whose group merges with LiveRamp, then also matches Experian |
| `v5` | Throtle-leftover whose group never merges with anything |
| `v6` | **Throtle-side ambiguous** — one UID claimed by three Throtle groups; graph-connected |
| `v13` | **LiveRamp-side ambiguous** — one UID claimed by two LiveRamp groups |
| `v14` | Member of the unmerged group `v13` also belongs to |
| `v15` | Experian-only UID that inherits a LiveRamp matchid through its group's alignment |
| `v16` | Experian-only UID with no alignment at all |
| `v17` | Reached by propagation, one hop from `v1` |
| `v18` | Reached by propagation, two hops (via `v17`) |
| `v20` | Reached by propagation from the ambiguous seed `v6` |
| `v19`, `v21` | Never resolved by providers or propagation; clustered together by Louvain |
| `v22`…`v27` | **Tie edge case** — one Throtle group (`T6`) overlapping two LiveRamp groups equally (`v22` in `L4`, `v23` in `L5`). `v24`/`v25`/`v26` are also wired to `v9` by an IP-collocation edge each — enough to make them graph-connected, not enough to inherit `v9`'s `L1`, since a UID already carrying a matchid is never a propagation candidate |
| `v30`, `v31` | **`LREX` case** — a LiveRamp-only group (`L6`) that merges directly with Experian, with no Throtle involvement. Only reachable because Job 1 parks `lr_out` in `lrth` |
| `v32` | Experian-only UID absorbed into that same `L6` merge |
| `v33`, `v34` | **Geo-mismatch edge case** — same weight as `v19`↔`v21`, but `v33` sits in `G3` and `v34` in `G4`. Never resolved by anything, ever |

## The four ways this pipeline produces more matchids than people

The phase doc groups the pipeline's edge cases into four mechanism families. Two are demonstrable at toy scale; two need production volume:

| Family | Mechanism | Shown here? |
|---|---|---|
| **A. Source-level ambiguity** | A vendor hands one UID to several groups and nothing collapses them | **Yes** — `v6` (Throtle side, 3 groups), `v13` (LiveRamp side, 2 groups) |
| **B. Arbitration failure** | Mutual-best rank refuses to merge, or merges only half | **Partly** — `T6`/`L4`/`L5` shows the `count1` tie and the stranded-loser variant. The `count2` collateral case and the Job 2 Experian tie need production data |
| **C. Reason-code fidelity** | A lexicographic `max(reason)` in Job 2 that *would* mislabel lineage | No — latent, and provably unreachable |
| **D. Assignment loss in propagation** | The per-matchid cap of ten is applied *after* each UID is reduced to one candidate, so UIDs are dropped while alternatives sit unused | No — needs a matchid with >10 contenders |

A useful non-case: a UID cannot be Throtle-ambiguous *and* LiveRamp-ambiguous at once. `th_only` excludes by UID, so the moment a UID appears anywhere in LiveRamp, all of its Throtle rows drop out. Worst-case fan-out for one UID is the max of the two, never the product.

In [1]:
import sys
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path.home() / ".pulsepoint"))
try:
    from dotenv import load_dotenv
    load_dotenv(Path.home() / ".pulsepoint" / ".env")
except ImportError:
    pass  # env may already be set in the jupyter process

import dbfuncs
import pandas as pd

S = "iceberg.jteixeira_ipa"

# Matchid typing matches production: bigint, not varchar. The type is
# load-bearing -- `order by matchid desc` decides propagation tie-breaks and
# must compare numerically, as it does in prod on murmur3 hashes.
#
# Matchid legend (written out literally everywhere below, no indirection):
#   Throtle   T1=101  T2=102  T3=103  T4=104  T5=105  T6=106  T9=109
#   LiveRamp  L1=201  L2=202  L3=203  L4=204  L5=205  L6=206
#   Experian  E1=301  E2=302  E3=303
#   Louvain   C1=901
#
# Values ascend with their labels, so every documented tie-break outcome
# (e.g. v20 -> T5, the largest of T3/T4/T5) still holds. Real hashes are often
# NEGATIVE, so in production that tie-break is arbitrary, not "highest label".

def q(sql):
    return dbfuncs.query2df(sql, print_time=False)

def make_table(sql, table, **kw):
    dbfuncs.query2table(sql, table, catalog_schema=S, print_time=False, **kw)
    return q(f"select * from {S}.{table}")

def insert_stage(sql, stage):
    """Idempotent stage append: clear the stage's rows first, then insert."""
    dbfuncs.run_query(f"delete from {S}.nscreen_toy_staged where stage = '{stage}'", print_time=False)
    dbfuncs.query2table(sql, "nscreen_toy_staged", catalog_schema=S, insert=True, print_time=False)
    return q(f"select uid, matchid, reason from {S}.nscreen_toy_staged where stage = '{stage}'")

def check(name, df, expected_rows, cols):
    """Multiset-compare a dataframe against expected tuples; print PASS/FAIL with diff.
    Counts matter: a duplicated row FAILs even if the value set matches."""
    got = Counter(map(tuple, df[cols].itertuples(index=False)))
    exp = Counter(expected_rows)
    if got == exp:
        print(f"PASS  {name} ({sum(exp.values())} rows)")
        return True
    print(f"FAIL  {name}")
    missing = list((exp - got).elements())
    unexpected = list((got - exp).elements())
    if missing:
        print("  missing:", sorted(missing))
    if unexpected:
        print("  unexpected/duplicated:", sorted(unexpected))
    return False

## Step 0 — (Re)create the four toy source tables

Four inputs, mirroring the four real sources: one table per vendor, plus the IP-collocation edge list that later phases propagate along.

Read these tables as *claims*, not facts. Each vendor row asserts "this cookie belongs to this person-group", and the vendors contradict each other — that disagreement is the entire subject of this notebook.

Two deliberate ambiguities are built in, because they are the dominant real-world failure mode (in production, 9.26% of LiveRamp UIDs and 0.41% of Throtle UIDs carry more than one matchid per day):

- `v6` appears under **three** Throtle matchids (`T3`, `T4`, `T5`).
- `v13` appears under **two** LiveRamp matchids (`L2`, `L3`).

**Plus the tie edge case:** Throtle group `T6` = {`v22`…`v27`}, where `v22` is also in LiveRamp group `L4` and `v23` in LiveRamp group `L5` — one Throtle group overlapping **two** LiveRamp groups with equal overlap (1 shared UID each). This probes whether alignment can inflate matchid cardinality: does `T6` merge into `L4`, `L5`, or neither?

Note also what the IP-collocation edges are *not*: they come from observed activity logs (`visitorlogevent` / `recordingpixellogevent`), never from vendor tables. They are independent evidence, which is why later phases can use them to reach UIDs no vendor resolved.

In [2]:
make_table("""
select * from (values
  ('v1',101), ('v2',101), ('v3',101),          -- T1
  ('v4',102),                                  -- T2
  ('v5',109),                                  -- T9
  ('v6',103), ('v6',104), ('v6',105),          -- T3, T4, T5  (one uid, three groups)
  ('v22',106), ('v23',106), ('v24',106),       -- T6
  ('v25',106), ('v26',106), ('v27',106)        -- T6
) as t(uid, matchid)
""", "nscreen_toy_th_source", recreate=True)

make_table("""
select * from (values
  ('v1',201), ('v2',201), ('v9',201),          -- L1
  ('v4',202), ('v10',202),                     -- L2
  ('v13',202), ('v13',203),                    -- L2 and L3  (one uid, two groups)
  ('v14',203),                                 -- L3
  ('v22',204),                                 -- L4
  ('v23',205),                                 -- L5
  ('v30',206), ('v31',206)                     -- L6: LiveRamp-only, no Throtle overlap
) as t(uid, matchid)
""", "nscreen_toy_lr_source", recreate=True)

make_table("""
select * from (values
  ('v3',301), ('v15',301),                     -- E1
  ('v16',302),                                 -- E2
  ('v30',303), ('v32',303)                     -- E3: shares only v30 with L6
) as t(uid, matchid)
""", "nscreen_toy_ex_source", recreate=True)

make_table("""
select * from (values
  ('v1','v17',5,false,false),
  ('v17','v18',3,false,false),
  ('v6','v20',2,false,false),
  ('v19','v21',4,false,false),
  -- v33/v34: same weight as v19/v21, but different geo buckets -- demonstrates
  -- that a geo mismatch kills a remainder edge exactly like having no edge at all.
  ('v33','v34',3,false,false),
  -- v9-v24/v25/v26: links a resolved LiveRamp-only member (v9, already L1) to three
  -- tie-rejected Throtle leftovers (v24/v25/v26, T6) -- all four already have a
  -- matchid by `initial`, so `connected_ns` picks all of them up as seeds, but none
  -- of them is a candidate for propagation (already assigned): v24/v25/v26 do NOT
  -- inherit L1.
  ('v9','v24',4,false,false),
  ('v9','v25',4,false,false),
  ('v9','v26',4,false,false)
) as t(uid1, uid2, weight, samedevice, differentusers)
""", "nscreen_toy_ic_source", recreate=True)

Dropping table iceberg.jteixeira_ipa.nscreen_toy_th_source
Query executed in 2.018 seconds
Creating table iceberg.jteixeira_ipa.nscreen_toy_th_source
Dropping table iceberg.jteixeira_ipa.nscreen_toy_lr_source
Query executed in 2.020 seconds
Creating table iceberg.jteixeira_ipa.nscreen_toy_lr_source
Dropping table iceberg.jteixeira_ipa.nscreen_toy_ex_source
Query executed in 1.991 seconds
Creating table iceberg.jteixeira_ipa.nscreen_toy_ex_source
Dropping table iceberg.jteixeira_ipa.nscreen_toy_ic_source
Query executed in 1.997 seconds
Creating table iceberg.jteixeira_ipa.nscreen_toy_ic_source


,uid1,uid2,weight,samedevice,differentusers
0,v1,v17,5,False,False
1,v17,v18,3,False,False
2,v6,v20,2,False,False
3,v19,v21,4,False,False
4,v33,v34,3,False,False
5,v9,v24,4,False,False
6,v9,v25,4,False,False
7,v9,v26,4,False,False


## Job 1 — Align LiveRamp and Throtle

Source: `src/nscreen_graph/spark/sparksql/NScreenLiverampThrotleOnlyStage.sql`.

The mutual-best `mapping` is computed first (CTEs `joined` → `grouped` → `ranked` → `limited` → `mapping`, verbatim logic). This is the algorithm from the section above, applied to the Throtle↔LiveRamp pair.

Work through the candidate pairings by hand before running the cell — the UID join produces only four:

| throtmatchid | lrmatchid | shared UIDs |
|---|---|---|
| `T1` | `L1` | 2 (`v1`, `v2`) |
| `T2` | `L2` | 1 (`v4`) |
| `T6` | `L4` | 1 (`v22`) |
| `T6` | `L5` | 1 (`v23`) |

`T1→L1` and `T2→L2` are each the only candidate on both of their sides → mutual-best → **accepted**.

**Edge case — `T6` must NOT appear in the mapping.** `T6` overlaps `L4` (via `v22`) and `L5` (via `v23`) with `cnt = 1` each: a tie. Because step 3 uses `rank()` and not `row_number()`, *both* pairings get `rnk_1 = 1`, and both also rank #1 within their own LiveRamp partitions — so both clear step 4 and enter `limited`. There, `count1 = 2` for `T6` fails the `count1 = 1` filter and **both pairings are rejected**. Neither `L4` nor `L5` wins.

Expected mapping, then: `T1 → L1` and `T2 → L2`, nothing else.

In [3]:
q(f"""
with joined as (
    select
        throt.uid,
        throt.matchid as throtmatchid,
        lr.matchid as lrmatchid
    from {S}.nscreen_toy_th_source as throt
    inner join {S}.nscreen_toy_lr_source as lr on throt.uid = lr.uid
),
grouped as (
    select throtmatchid, lrmatchid, count(*) as cnt
    from joined
    group by throtmatchid, lrmatchid
),
ranked as (
    select
        throtmatchid, lrmatchid,
        rank() over (partition by throtmatchid order by cnt desc) as rnk_1,
        rank() over (partition by lrmatchid   order by cnt desc) as rnk_2
    from grouped
)
select * from ranked""")

,throtmatchid,lrmatchid,rnk_1,rnk_2
0,106,204,1,1
1,101,201,1,1
2,102,202,1,1
3,106,205,1,1


In [4]:
q(f"""
with joined as (
    select
        throt.uid,
        throt.matchid as throtmatchid,
        lr.matchid as lrmatchid
    from {S}.nscreen_toy_th_source as throt
    inner join {S}.nscreen_toy_lr_source as lr on throt.uid = lr.uid
),
grouped as (
    select throtmatchid, lrmatchid, count(*) as cnt
    from joined
    group by throtmatchid, lrmatchid
),
ranked as (
    select
        throtmatchid, lrmatchid,
        rank() over (partition by throtmatchid order by cnt desc) as rnk_1,
        rank() over (partition by lrmatchid   order by cnt desc) as rnk_2
    from grouped
),
limited as (
    select
        throtmatchid, lrmatchid,
        count(*) over (partition by throtmatchid) as count1,
        count(*) over (partition by lrmatchid)   as count2
    from ranked
    where rnk_1 = 1 and rnk_2 = 1
)
select *
from limited""")

,throtmatchid,lrmatchid,count1,count2
0,106,204,2,1
1,101,201,1,1
2,102,202,1,1
3,106,205,2,1


In [5]:
df = make_table(f"""
with joined as (
    select
        throt.uid,
        throt.matchid as throtmatchid,
        lr.matchid as lrmatchid
    from {S}.nscreen_toy_th_source as throt
    inner join {S}.nscreen_toy_lr_source as lr on throt.uid = lr.uid
),
grouped as (
    select throtmatchid, lrmatchid, count(*) as cnt
    from joined
    group by throtmatchid, lrmatchid
),
ranked as (
    select
        throtmatchid, lrmatchid,
        rank() over (partition by throtmatchid order by cnt desc) as rnk_1,
        rank() over (partition by lrmatchid   order by cnt desc) as rnk_2
    from grouped
),
limited as (
    select
        throtmatchid, lrmatchid,
        count(*) over (partition by throtmatchid) as count1,
        count(*) over (partition by lrmatchid)   as count2
    from ranked
    where rnk_1 = 1 and rnk_2 = 1
)
select throtmatchid, lrmatchid
from limited
where count1 = 1 and count2 = 1
""", "nscreen_toy_mapping_th_lr", recreate=True)

check("Job1 mapping", df, {
    (101, 201),   # T1 -> L1
    (102, 202),   # T2 -> L2
}, ["throtmatchid", "lrmatchid"])
df

Dropping table iceberg.jteixeira_ipa.nscreen_toy_mapping_th_lr
Query executed in 1.979 seconds
Creating table iceberg.jteixeira_ipa.nscreen_toy_mapping_th_lr
PASS  Job1 mapping (2 rows)


,throtmatchid,lrmatchid
0,101,201
1,102,202


In [6]:
q("select * from iceberg.jteixeira_ipa.nscreen_toy_mapping_th_lr")

,throtmatchid,lrmatchid
0,101,201
1,102,202


### The two output branches: `lr_out` and `th_out`

Job 1 emits two row sets, and the asymmetry between them is the single most consequential design choice in the pipeline.

**`lr_out`** — a pure pass-through of *every* LiveRamp row, matchid kept verbatim, with the reason set by whether that matchid was in the mapping (`LRTH` if merged, `LR` if not). There is no per-UID filter at all, which is why LiveRamp-side ambiguity survives untouched.

**`th_out`** — Throtle rows whose UID **never appears anywhere in `liveramp_source`**, with the matchid translated via `coalesce(m.lrmatchid, t.matchid)`. That existence check (`th_only`) is per-UID, not per-group: it filters by whether the *cookie* was seen in LiveRamp, while the merge it applies was decided per *matchid*. Those two different granularities cause most of the surprising behaviour downstream.

Consequences to watch for in the output:

- `v1`, `v2`, `v22`, `v23` vanish from `th_out` even though they belong to Throtle groups — they appear in LiveRamp, so `th_only` drops them. Their Throtle group membership still mattered, though: it was the evidence that earned the merge.
- `v3` gets matchid `L1` with reason `LRTH` despite having **no LiveRamp row of its own**. Its group `T1` merged, and the translation applies to the whole group. Group-level inheritance, not UID-level matching.
- `v6`'s three rows all survive independently, each translated (or not) on its own — 3 rows, 3 matchids, one cookie.

Expected `lr_out`: 10 rows — `v13` twice (`L2`/LRTH and `L3`/LR), plus the edge case `v22`→`L4`/**LR** and `v23`→`L5`/**LR** (their `T6` pairing was tie-rejected, so no LRTH).

Expected `th_out`: 9 rows — `v3`→`L1`/LRTH, `v5`→`T9`/TH, `v6` three times (`T3`, `T4`, `T5`, all TH), and `v24`…`v27` keeping `T6`/TH (T6 unmapped after the tie rejection).

In [7]:
lr_out = make_table(f"""
select
    lr.uid,
    lr.matchid,
    case when m.lrmatchid is not null then 'LRTH' else 'LR' end as reason
from {S}.nscreen_toy_lr_source as lr
left join {S}.nscreen_toy_mapping_th_lr as m on lr.matchid = m.lrmatchid
""", "nscreen_toy_lr_out", recreate=True)

check("lr_out", lr_out, [
    ("v1", 201, "LRTH"), ("v2", 201, "LRTH"), ("v9", 201, "LRTH"),   # L1 merged
    ("v4", 202, "LRTH"), ("v10", 202, "LRTH"), ("v13", 202, "LRTH"), # L2 merged
    ("v13", 203, "LR"), ("v14", 203, "LR"),                          # L3 unmerged
    ("v22", 204, "LR"),                                              # L4 tie-rejected
    ("v23", 205, "LR"),                                              # L5 tie-rejected
    ("v30", 206, "LR"), ("v31", 206, "LR"),                          # L6 no Throtle
], ["uid", "matchid", "reason"])
lr_out

Dropping table iceberg.jteixeira_ipa.nscreen_toy_lr_out
Query executed in 1.485 seconds
Creating table iceberg.jteixeira_ipa.nscreen_toy_lr_out
PASS  lr_out (12 rows)


,uid,matchid,reason
0,v30,206,LR
1,v31,206,LR
2,v13,203,LR
3,v14,203,LR
4,v1,201,LRTH
5,v2,201,LRTH
6,v9,201,LRTH
7,v4,202,LRTH
8,v10,202,LRTH
9,v13,202,LRTH


In [8]:

th_out = make_table(f"""
with th_only as (
    select t.*
    from {S}.nscreen_toy_th_source as t
    left join {S}.nscreen_toy_lr_source as lr on t.uid = lr.uid
    where lr.uid is null
)
select
    t.uid,
    coalesce(m.lrmatchid, t.matchid) as matchid,
    case when m.throtmatchid is not null then 'LRTH' else 'TH' end as reason
from th_only as t
left join {S}.nscreen_toy_mapping_th_lr as m on t.matchid = m.throtmatchid
""", "nscreen_toy_th_out", recreate=True)

check("th_out", th_out, [
    ("v3", 201, "LRTH"),                                     # inherits T1's merge -> L1
    ("v5", 109, "TH"),                                       # T9
    ("v6", 103, "TH"), ("v6", 104, "TH"), ("v6", 105, "TH"), # T3, T4, T5
    ("v24", 106, "TH"), ("v25", 106, "TH"),                  # T6 kept its own id
    ("v26", 106, "TH"), ("v27", 106, "TH"),
], ["uid", "matchid", "reason"])

th_out

Dropping table iceberg.jteixeira_ipa.nscreen_toy_th_out
Query executed in 1.978 seconds
Creating table iceberg.jteixeira_ipa.nscreen_toy_th_out
PASS  th_out (9 rows)


,uid,matchid,reason
0,v3,201,LRTH
1,v27,106,TH
2,v6,105,TH
3,v5,109,TH
4,v6,104,TH
5,v6,103,TH
6,v25,106,TH
7,v26,106,TH
8,v24,106,TH


### Job 1's write: everything lands in `lrth`

Here is the step that makes this a replication of **production** rather than of the SQL text.

Job 1's SQL labels `lr_out` as `stage = 'initial'` and `th_out` as `stage = 'lrth'` — but those literals never survive. `common.py:49-51` overwrites every partition column with the value the Luigi task supplies, and Job 1 supplies `("stage", "lrth")`:

```python
for col_name, col_value in args.partition_columns:
    df = df.withColumn(col_name, F.lit(col_value))
```

So both branches land in one partition. The table below stands in for that real partition, and Job 2 reads it whole — which is why Experian ends up compared against LiveRamp-rooted groups, and why the reason `LREX` exists in production at all (~1.6B rows/day).

Expected: 21 rows — `th_out`'s 9 plus `lr_out`'s 12.

In [9]:
# Production's real stage='lrth' contents: Job 1's ENTIRE output, both branches.
lrth_all = make_table(f"""
select uid, matchid, reason from {S}.nscreen_toy_th_out
union all
select uid, matchid, reason from {S}.nscreen_toy_lr_out
""", "nscreen_toy_lrth_all", recreate=True)

print(f"lrth_all: {len(lrth_all)} rows "
      f"({len(th_out)} from th_out + {len(lr_out)} from lr_out)")
lrth_all.sort_values(["uid", "matchid"])

Dropping table iceberg.jteixeira_ipa.nscreen_toy_lrth_all
Query executed in 1.983 seconds
Creating table iceberg.jteixeira_ipa.nscreen_toy_lrth_all
lrth_all: 21 rows (9 from th_out + 12 from lr_out)


,uid,matchid,reason
4,v1,201,LRTH
8,v10,202,LRTH
9,v13,202,LRTH
2,v13,203,LR
3,v14,203,LR
5,v2,201,LRTH
11,v22,204,LR
10,v23,205,LR
16,v24,106,TH
20,v25,106,TH


## Job 2 — Add Experian

Source: `src/nscreen_graph/spark/sparksql/NScreenLiverampThrotleExInitialStage.sql`.

Identical mutual-best-rank algorithm, now pairing Experian matchids against the matchids Job 1 established. Experian is the easiest of the three vendors to align: it never assigns one UID to multiple matchids (measured at 0% ambiguity in production), so it contributes no fan-out of its own — the only way Experian evidence gets lost is a tie rejection.

Its input is the **whole** `lrth` partition (`nscreen_toy_lrth_all`), which is what production actually feeds it. Two pairings come out of the toy data:

| exmatchid | lrthmatchid | via | that group's reason | result |
|---|---|---|---|---|
| `E1` | `L1` | `v3` | `LRTH` (Throtle-derived) | → `LRTHEX` |
| `E3` | `L6` | `v30` | `LR` (**LiveRamp-only**) | → **`LREX`** |

The second row is the one that cannot happen if you believe the SQL's `stage` literals — and it is the second-most-common reason in production.

Two branches, mirroring Job 1:

- **`lrth_out`** — every input row rewritten, matchid **unchanged**, reason gaining an `EX` suffix where its matchid matched an Experian group. Job 2 never moves a row to a new matchid here; it only annotates.
- **`ex_only` / `ex_out`** — Experian rows whose UID does not appear in the input at all, translated into the aligned namespace where possible (`coalesce`), else keeping their own Experian matchid with reason `EX`.

Watch for two things in the output:

1. **The `EX` suffix spreads to the whole group.** `L1` earned its Experian merge through `v3` alone, yet `v1`, `v2` and `v9` all come out `LRTHEX` — none of them has an Experian row. The merge is decided per matchid and applied per matchid.
2. **A UID's own Experian row can be pure evidence.** `v3` and `v30` are in `ex_source` *and* in the input, so `ex_only` excludes them — their Experian membership is consumed by the algorithm and never becomes an output row of its own.

In [10]:
q(f"""with joined as (
    select
        ex.uid,
        ex.matchid as exmatchid,
        lrth.matchid as lrthmatchid,
        lrth.reason
    from {S}.nscreen_toy_ex_source as ex
    inner join {S}.nscreen_toy_lrth_all as lrth on ex.uid = lrth.uid
),
grouped as (
    select exmatchid, lrthmatchid, max(reason) as reason, count(*) as cnt
    from joined
    group by exmatchid, lrthmatchid
),
ranked as (
    select
        exmatchid, lrthmatchid, reason,
        rank() over (partition by exmatchid   order by cnt desc) as rnk_1,
        rank() over (partition by lrthmatchid order by cnt desc) as rnk_2
    from grouped
)
select * from ranked
""")

,exmatchid,lrthmatchid,reason,rnk_1,rnk_2
0,301,201,LRTH,1,1
1,303,206,LR,1,1


In [11]:
df = make_table(f"""
with joined as (
    select
        ex.uid,
        ex.matchid as exmatchid,
        lrth.matchid as lrthmatchid,
        lrth.reason
    from {S}.nscreen_toy_ex_source as ex
    inner join {S}.nscreen_toy_lrth_all as lrth on ex.uid = lrth.uid
),
grouped as (
    select exmatchid, lrthmatchid, max(reason) as reason, count(*) as cnt
    from joined
    group by exmatchid, lrthmatchid
),
ranked as (
    select
        exmatchid, lrthmatchid, reason,
        rank() over (partition by exmatchid   order by cnt desc) as rnk_1,
        rank() over (partition by lrthmatchid order by cnt desc) as rnk_2
    from grouped
),
limited as (
    select
        exmatchid, lrthmatchid, reason,
        count(*) over (partition by exmatchid)   as count1,
        count(*) over (partition by lrthmatchid) as count2
    from ranked
    where rnk_1 = 1 and rnk_2 = 1
)
select exmatchid, lrthmatchid, reason
from limited
where count1 = 1 and count2 = 1
""", "nscreen_toy_mapping_lrth_ex", recreate=True)

# E1->L1 via v3 (a Throtle-derived group, reason LRTH).
# E3->L6 via v30 (a pure LiveRamp group, reason LR) -> this is what yields LREX,
# and it is only reachable because Job 1 parked lr_out in 'lrth' too.
check("Job2 mapping", df, {
    (301, 201, "LRTH"),   # E1 -> L1, via v3 (a Throtle-derived group)
    (303, 206, "LR"),     # E3 -> L6, via v30 (a pure LiveRamp group) -> yields LREX
}, ["exmatchid", "lrthmatchid", "reason"])
df

Dropping table iceberg.jteixeira_ipa.nscreen_toy_mapping_lrth_ex
Query executed in 1.786 seconds
Creating table iceberg.jteixeira_ipa.nscreen_toy_mapping_lrth_ex
PASS  Job2 mapping (2 rows)


,exmatchid,lrthmatchid,reason
0,301,201,LRTH
1,303,206,LR


In [12]:
lrth_out = make_table(f"""
select
    lrth.uid,
    lrth.matchid,
    case when m.lrthmatchid is not null then lrth.reason || 'EX' else lrth.reason end as reason
from {S}.nscreen_toy_lrth_all as lrth
left join {S}.nscreen_toy_mapping_lrth_ex as m on lrth.matchid = m.lrthmatchid
""", "nscreen_toy_lrth_out", recreate=True)

check("lrth_out", lrth_out, [
    # L1 (201) merged with Experian, so EVERY L1 member gains EX -- including
    # v1/v2/v9, which have no Experian row of their own. Matchid-level, not uid-level.
    ("v1", 201, "LRTHEX"), ("v2", 201, "LRTHEX"), ("v9", 201, "LRTHEX"),
    ("v3", 201, "LRTHEX"),
    # L2 (202) / L3 (203) / L4 (204) / L5 (205): no Experian match, reasons unchanged
    ("v4", 202, "LRTH"), ("v10", 202, "LRTH"), ("v13", 202, "LRTH"),
    ("v13", 203, "LR"), ("v14", 203, "LR"),
    ("v22", 204, "LR"), ("v23", 205, "LR"),
    # L6 (206) is LiveRamp-only and merged with Experian -> LREX
    ("v30", 206, "LREX"), ("v31", 206, "LREX"),
    # Throtle leftovers, no Experian match
    ("v5", 109, "TH"),
    ("v6", 103, "TH"), ("v6", 104, "TH"), ("v6", 105, "TH"),
    ("v24", 106, "TH"), ("v25", 106, "TH"), ("v26", 106, "TH"), ("v27", 106, "TH"),
], ["uid", "matchid", "reason"])

ex_out = make_table(f"""
with ex_only as (
    select e.*
    from {S}.nscreen_toy_ex_source as e
    left join {S}.nscreen_toy_lrth_all as lrth on e.uid = lrth.uid
    where lrth.uid is null
)
select
    e.uid,
    coalesce(m.lrthmatchid, e.matchid) as matchid,
    case when m.exmatchid is not null then m.reason || 'EX' else 'EX' end as reason
from ex_only as e
left join {S}.nscreen_toy_mapping_lrth_ex as m on e.matchid = m.exmatchid
""", "nscreen_toy_ex_out", recreate=True)

check("ex_out", ex_out, [
    ("v15", 201, "LRTHEX"),   # inherits L1 through E1's alignment
    ("v16", 302, "EX"),       # E2, no alignment at all
    ("v32", 206, "LREX"),     # absorbed into the LiveRamp-only merged group L6
], ["uid", "matchid", "reason"])
ex_out

Dropping table iceberg.jteixeira_ipa.nscreen_toy_lrth_out
Query executed in 2.036 seconds
Creating table iceberg.jteixeira_ipa.nscreen_toy_lrth_out
PASS  lrth_out (21 rows)
Dropping table iceberg.jteixeira_ipa.nscreen_toy_ex_out
Query executed in 2.064 seconds
Creating table iceberg.jteixeira_ipa.nscreen_toy_ex_out
PASS  ex_out (3 rows)


,uid,matchid,reason
0,v15,201,LRTHEX
1,v32,206,LREX
2,v16,302,EX


In [13]:
q("select * from iceberg.jteixeira_ipa.nscreen_toy_lrth_out")

,uid,matchid,reason
0,v30,206,LREX
1,v31,206,LREX
2,v13,203,LR
3,v14,203,LR
4,v1,201,LRTHEX
5,v2,201,LRTHEX
6,v9,201,LRTHEX
7,v4,202,LRTH
8,v10,202,LRTH
9,v13,202,LRTH


In [14]:
q("select * from iceberg.jteixeira_ipa.nscreen_toy_ex_out")

,uid,matchid,reason
0,v15,201,LRTHEX
1,v32,206,LREX
2,v16,302,EX


## Assemble the staged table (`lrth` + `initial`)

Mirrors the real single-table design: everything lands in one table distinguished by a `stage` column, and later phases append rather than overwrite.

Following production (not the SQL text):

- **`lrth`** = *all* of Job 1's output — `th_out` ∪ `lr_out`.
- **`initial`** = *all* of Job 2's output — `lrth_out` (every `lrth` row rewritten) ∪ `ex_out`.

So every LiveRamp-rooted row appears **twice**: once in `lrth` as Job 1 left it, once in `initial` as Job 2 re-stamped it. That is why per-UID `distinct_stage` is 2 for ordinary LiveRamp UIDs here, and it is a lineage artefact — not ambiguity.

Two things to notice as you read the assembled table:

- A row appearing at two stages is **not** a sign of ambiguity — it is ordinary lineage, the same row carried forward and re-annotated. The real ambiguity signal is **one UID carrying two different matchids**, which can happen within a single stage (`v13`).
- No step anywhere in provider alignment does `group by uid`. `th_only` filters by UID *existence*; every translation joins by *matchid*; the final result job is a pure filter. The mutual-best logic arbitrates between group *pairings*, never between several groups' competing claims on one cookie. That is why `v6` keeps three matchids and `v13` keeps two, all the way into the result.

In [15]:
# Production semantics: ALL of Job 1's output is parked at stage='lrth'
# (its 'initial' literal is discarded), and stage='initial' is Job 2's output alone.
staged = make_table(f"""
select 'lrth' as stage, uid, matchid, reason from {S}.nscreen_toy_lrth_all
union all
select 'initial' as stage, uid, matchid, reason from {S}.nscreen_toy_lrth_out
union all
select 'initial' as stage, uid, matchid, reason from {S}.nscreen_toy_ex_out
""", "nscreen_toy_staged", recreate=True)

staged.groupby("stage").size()

Dropping table iceberg.jteixeira_ipa.nscreen_toy_staged
Query executed in 1.876 seconds
Creating table iceberg.jteixeira_ipa.nscreen_toy_staged


stage
initial    24
lrth       21
dtype: int64

In [16]:
staged.sort_values(by=['stage', 'uid'], ascending=[False, True]).reset_index(drop=True)

,stage,uid,matchid,reason
0,lrth,v1,201,LRTH
1,lrth,v10,202,LRTH
2,lrth,v13,203,LR
3,lrth,v13,202,LRTH
4,lrth,v14,203,LR
5,lrth,v2,201,LRTH
6,lrth,v22,204,LR
7,lrth,v23,205,LR
8,lrth,v24,106,TH
9,lrth,v25,106,TH


## Phase 5a — connected_ns

Source: `src/nscreen_graph/spark/sparksql/NScreenLiverampThrotleExConnectedNsStage.sql`.

From here on the pipeline stops asking vendors and starts using observed behaviour. This step selects the **seeds** for propagation: every `initial` row whose UID appears anywhere in the IP-collocation edge table, stamped with reason `NA` (a placeholder — these copies never reach the final result).

The join is on UID with an inner join, so it *multiplies* rows: an ambiguous UID contributes all of its rows. Watch `v6` produce three seed rows, one per matchid — meaning the propagation that follows starts from a seed that is itself unresolved.

Expected: `v1` (1 row), `v6` (3 rows), `v9` (1 row), and `v24`/`v25`/`v26` (1 row each) — the last four newly graph-connected by the `v9`↔`v24`/`v25`/`v26` edges, even though every one of them already has a matchid.

In [17]:
q(f"""select uid1 as uid from {S}.nscreen_toy_ic_source
    union
    select uid2 as uid from {S}.nscreen_toy_ic_source""")

,uid
0,v1
1,v33
2,v17
3,v19
4,v25
5,v26
6,v18
7,v20
8,v34
9,v6


In [18]:
df = insert_stage(f"""
with distinct_ns_uids as (
    select uid1 as uid from {S}.nscreen_toy_ic_source
    union
    select uid2 as uid from {S}.nscreen_toy_ic_source
)
select
    'connected_ns' as stage,
    i.uid,
    i.matchid,
    'NA' as reason
from {S}.nscreen_toy_staged as i
inner join distinct_ns_uids as n on i.uid = n.uid
where i.stage = 'initial'
""", "connected_ns")

check("connected_ns", df, [
    ("v1", 201, "NA"),                                       # L1
    ("v6", 103, "NA"), ("v6", 104, "NA"), ("v6", 105, "NA"), # T3, T4, T5 -- all copied
    ("v9", 201, "NA"),                                       # L1 -- newly graph-connected via v24/v25/v26
    ("v24", 106, "NA"), ("v25", 106, "NA"), ("v26", 106, "NA"), # T6 -- newly graph-connected via v9
], ["uid", "matchid", "reason"])
df

Inserting into table iceberg.jteixeira_ipa.nscreen_toy_staged
PASS  connected_ns (8 rows)


,uid,matchid,reason
0,v24,106,NA
1,v1,201,NA
2,v26,106,NA
3,v25,106,NA
4,v6,105,NA
5,v6,104,NA
6,v6,103,NA
7,v9,201,NA


## Phase 5b — stage1 and stage2 propagation

Source: `src/nscreen_graph/spark/sparksql/NScreenLiverampThrotleExNStage.sql`, run twice.

Now matchids spread outward along IP-collocation edges to UIDs no vendor could resolve. Each round: double every edge so it points both ways, sum edge weight per `(uid, matchid)` candidate, drop UIDs that already have a matchid, keep **one winner per UID** (`row_number() … order by weight desc, matchid desc`), and finally cap each matchid at ten new UIDs per round.

This is the first point in the pipeline where **exactly one matchid per UID is guaranteed** — and note how it is achieved: by an arbitrary tie-break, not by evidence. An ambiguous seed stays ambiguous itself, but anything reached *from* it gets a single id chosen by sort order. `v20` demonstrates this precisely: it is reachable from `v6`'s three matchids at weight 2 each, a perfect three-way tie, resolved by `matchid desc` alone.

Toy adaptations: edge doubling via `union all` (Trino has no `inline(named_struct)`), and no `stable` weight-doubling (the toy has no `stable` column — in production, LiveRamp-derived candidates get 2× weight, biasing propagation toward LiveRamp *before* the tie-break applies).

Expected stage1: `v17`→`L1`, and `v20`→`T5` (the three-way tie broken by `order by weight desc, matchid desc`).
Expected stage2: `v18`→`L1` only — `v1` and `v6` are re-reached as candidates but dropped by `filter_out_having_matchid`, which is what makes the stages mutually exclusive. `v9`↔`v24`/`v25`/`v26` never even produce a candidate row in either round, for the same reason: all four endpoints already have a matchid before propagation starts.

Not reproducible at this scale: the ten-UID cap. In production 3.1M matchids saturate it per round, and because the cap is applied *after* each UID has been reduced to a single candidate, UIDs get dropped entirely while their discarded alternatives still have free slots. See [`provider-alignment-defects.md`](docs/analysis/provider-alignment-defects.md).

> **Observed versus expected.** Current cells reproduce production behavior: `v9` remains `L1`, while `v24`/`v25`/`v26` remain `T6`. Their edges only create `connected_ns` copies because `filter_out_having_matchid` removes every already-assigned UID before ranking.
>
> Expected behavior is provider-neutral: sufficiently strong IP-collocation or first-party evidence, supported by `samedevice` and `differentusers` signals, should place `v9` and `v24`/`v25`/`v26` under one canonical matchid. Similar reconciliation should prevent same-vendor ambiguity such as `v6` carrying `T3`/`T4`/`T5` or `v13` carrying `L2`/`L3`. Every published UID should map to exactly one matchid; one matchid may map to many UIDs. This notebook does not implement that expected behavior; see [`matchid-collocation-reconciliation-plan.md`](docs/plans/matchid-collocation-reconciliation-plan.md).

In [19]:
def propagation_sql(stage_label, seed_stages):
    seeds = ", ".join(f"'{s}'" for s in seed_stages)
    return f"""
with ns_dbl as (
    select uid1, uid2, weight from {S}.nscreen_toy_ic_source
    union all
    select uid2 as uid1, uid1 as uid2, weight from {S}.nscreen_toy_ic_source
),
connected_ns as (
    select uid, matchid
    from {S}.nscreen_toy_staged
    where stage in ({seeds})
),
joined_by_uid2 as (
    select ns_dbl.uid1, ns_dbl.weight, connected_ns.matchid
    from ns_dbl
    inner join connected_ns on ns_dbl.uid2 = connected_ns.uid
),
grouped as (
    select uid1 as uid, matchid, sum(weight) as weight
    from joined_by_uid2
    group by uid1, matchid
),
filter_out_having_matchid as (
    select g.uid, g.matchid, g.weight
    from grouped as g
    left join connected_ns as c on g.uid = c.uid
    where c.matchid is null
),
ranked_by_uid as (
    select uid, matchid, weight,
           row_number() over (partition by uid order by weight desc, matchid desc) as rnk
    from filter_out_having_matchid
),
filtered_by_uid_rnk as (
    select uid, matchid, weight from ranked_by_uid where rnk = 1
),
ranked_by_matchid as (
    select uid, matchid,
           row_number() over (partition by matchid order by weight desc, uid desc) as rnk
    from filtered_by_uid_rnk
)
select '{stage_label}' as stage, uid, matchid, 'IC' as reason
from ranked_by_matchid
where rnk <= 10
"""

Before running the full round, look at the two intermediate CTEs that decide the outcome — same pattern as Job 1's `ranked` → `limited` above.

In [20]:
# `filter_out_having_matchid`: every (uid, matchid) candidate for stage1,
# with edge weight summed, BEFORE the uid winner or the matchid cap is applied.
q(f"""
with ns_dbl as (
    select uid1, uid2, weight from {S}.nscreen_toy_ic_source
    union all
    select uid2 as uid1, uid1 as uid2, weight from {S}.nscreen_toy_ic_source
),
connected_ns as (
    select uid, matchid from {S}.nscreen_toy_staged where stage in ('connected_ns')
),
joined_by_uid2 as (
    select ns_dbl.uid1, ns_dbl.weight, connected_ns.matchid
    from ns_dbl inner join connected_ns on ns_dbl.uid2 = connected_ns.uid
),
grouped as (
    select uid1 as uid, matchid, sum(weight) as weight
    from joined_by_uid2 group by uid1, matchid
)
select g.uid, g.matchid, g.weight
from grouped as g
left join connected_ns as c on g.uid = c.uid
where c.matchid is null
""").sort_values(["uid", "matchid"])

,uid,matchid,weight
0,v17,201,5
3,v20,103,2
1,v20,104,2
2,v20,105,2


`v17` has one candidate. `v20` has three, tied at weight 2 — reached from `v6`'s three matchids (`T3`=103, `T4`=104, `T5`=105) via the single edge `v6`↔`v20`. `ranked_by_uid` is where that tie actually gets broken:

In [21]:
# `ranked_by_uid`: exactly one candidate kept per UID (rnk = 1) -- this is where
# v20's three-way tie collapses to a single matchid, by `matchid desc` alone.
ranked_by_uid = q(f"""
with ns_dbl as (
    select uid1, uid2, weight from {S}.nscreen_toy_ic_source
    union all
    select uid2 as uid1, uid1 as uid2, weight from {S}.nscreen_toy_ic_source
),
connected_ns as (
    select uid, matchid from {S}.nscreen_toy_staged where stage in ('connected_ns')
),
joined_by_uid2 as (
    select ns_dbl.uid1, ns_dbl.weight, connected_ns.matchid
    from ns_dbl inner join connected_ns on ns_dbl.uid2 = connected_ns.uid
),
grouped as (
    select uid1 as uid, matchid, sum(weight) as weight
    from joined_by_uid2 group by uid1, matchid
),
filter_out_having_matchid as (
    select g.uid, g.matchid, g.weight
    from grouped as g
    left join connected_ns as c on g.uid = c.uid
    where c.matchid is null
)
select uid, matchid, weight,
       row_number() over (partition by uid order by weight desc, matchid desc) as rnk
from filter_out_having_matchid
""")
ranked_by_uid.sort_values(["uid", "rnk"])

,uid,matchid,weight,rnk
0,v17,201,5,1
1,v20,105,2,1
2,v20,104,2,2
3,v20,103,2,3


`filtered_by_uid_rnk` keeps only `rnk = 1` from the table above, and `ranked_by_matchid` + `rnk <= 10` applies the per-matchid cap (§D1/D2 in the phase doc — not reachable at this scale, since no toy matchid gets more than one winner). What's left is the actual stage1/stage2 output:

In [22]:
df = insert_stage(propagation_sql("stage1", ["connected_ns"]), "stage1")
# v20 is a three-way tie (T3=103 / T4=104 / T5=105, weight 2 each), broken by
# `matchid desc` -- numerically on bigint, exactly as production compares it.
check("stage1", df, [
    ("v17", 201, "IC"),   # L1, one hop from v1
    ("v20", 105, "IC"),   # T5 wins the tie as the largest matchid
], ["uid", "matchid", "reason"])

df = insert_stage(propagation_sql("stage2", ["connected_ns", "stage1"]), "stage2")
check("stage2", df, [("v18", 201, "IC")], ["uid", "matchid", "reason"])  # L1

Inserting into table iceberg.jteixeira_ipa.nscreen_toy_staged
PASS  stage1 (2 rows)
Inserting into table iceberg.jteixeira_ipa.nscreen_toy_staged
PASS  stage2 (1 rows)


True

## Phase 6 — remainder + (simulated) Louvain

Remainder source: `src/nscreen_graph/spark/sparksql/NScreenIpCollocationDailyExRemainder.sql`.

The last resort, for UIDs that no vendor claimed and no propagation reached. An edge is kept only when **both** endpoints are still unassigned (`stage in ('initial','stage1','stage2')`) *and* the two endpoints share a geo bucket — then the surviving graph is clustered and each component becomes a synthetic community id.

Because both endpoints must be unassigned, this path is disjoint from everything before it: Louvain can only ever assign UIDs the earlier stages never touched.

Two ways an edge fails to reach Louvain, both demonstrated in the toy data:

- **No edge at all.** An edge-less UID never gets a Louvain assignment — nothing to check the geo of.
- **A geo mismatch.** `v33`↔`v34` has the same weight as `v19`↔`v21` (3 vs 4 — both survive the "unassigned" filter identically), but `v33` sits in geo `G3` and `v34` in `G4`. `where g1.geo = g2.geo` drops the row. The edge existed, both endpoints were eligible, and it still never counts — a mismatched edge is functionally identical to no edge at all.

Expected surviving edge: only `v19`↔`v21`. `v33` and `v34` get **no matchid, in no stage, ever** — not even a `NA` placeholder, since they never appear in `initial` for `connected_ns` to copy.

Clustering (`NScreenExRemainderLouvainStage.sql`) is **simulated**: the real UDAF `hudf_calc_screen7_ids_udaf` is unavailable in Trino, so both endpoints of the surviving component get `901` (valid because the toy remainder is a single 2-node component with no `differentusers` conflict).

In [23]:
make_table(f"""
select * from (values
  ('v33', 'G3'),
  ('v34', 'G4')
) as t(uid, geo)
union
select uid1 as uid, 'G1' as geo from {S}.nscreen_toy_ic_source
where uid1 not in ('v33', 'v34')
union
select uid2 as uid, 'G1' as geo from {S}.nscreen_toy_ic_source
where uid2 not in ('v33', 'v34')
""", "nscreen_toy_geo", recreate=True)

remainder = make_table(f"""
with uids_in_initial_and_stages as (
    select uid from {S}.nscreen_toy_staged
    where stage in ('initial', 'stage1', 'stage2')
),
collocation_minus_already_assigned as (
    select c.uid1, c.uid2, c.weight, c.samedevice, c.differentusers
    from {S}.nscreen_toy_ic_source as c
    left join uids_in_initial_and_stages as u1 on c.uid1 = u1.uid
    left join uids_in_initial_and_stages as u2 on c.uid2 = u2.uid
    where u1.uid is null and u2.uid is null
)
select c.uid1, c.uid2, c.weight, g1.geo
from collocation_minus_already_assigned as c
inner join {S}.nscreen_toy_geo as g1 on c.uid1 = g1.uid
inner join {S}.nscreen_toy_geo as g2 on c.uid2 = g2.uid
where g1.geo = g2.geo
""", "nscreen_toy_remainder", recreate=True)

# v33/v34 pass the "both unassigned" filter (same shape as v19/v21) but are
# dropped by the geo join -- confirming the mismatch, not the assignment check, is
# what excludes them.
check("remainder edges", remainder, [("v19", "v21")], ["uid1", "uid2"])

# SIMULATED louvain: every uid on a surviving edge -> community 901
df = insert_stage(f"""
with remainder_uids as (
    select uid1 as uid from {S}.nscreen_toy_remainder
    union
    select uid2 as uid from {S}.nscreen_toy_remainder
)
select 'louvain' as stage, uid, 901 as matchid, 'LU' as reason
from remainder_uids
""", "louvain")

check("louvain (simulated)", df, [
    ("v19", 901, "LU"), ("v21", 901, "LU"),   # 901 = synthetic community id
], ["uid", "matchid", "reason"])

# v33/v34 never appear in ANY stage, in the whole pipeline -- not even NA.
never_assigned = q(f"""
select uid from {S}.nscreen_toy_staged where uid in ('v33', 'v34')
""")
assert len(never_assigned) == 0, f"v33/v34 unexpectedly assigned: {never_assigned}"
print("PASS  v33/v34: zero rows in nscreen_toy_staged, in any stage")

Dropping table iceberg.jteixeira_ipa.nscreen_toy_geo
Query executed in 1.910 seconds
Creating table iceberg.jteixeira_ipa.nscreen_toy_geo
Dropping table iceberg.jteixeira_ipa.nscreen_toy_remainder
Query executed in 1.997 seconds
Creating table iceberg.jteixeira_ipa.nscreen_toy_remainder
PASS  remainder edges (1 rows)
Inserting into table iceberg.jteixeira_ipa.nscreen_toy_staged
PASS  louvain (simulated) (2 rows)
PASS  v33/v34: zero rows in nscreen_toy_staged, in any stage


In [24]:
remainder

,uid1,uid2,weight,geo
0,v19,v21,4,G1


## Final result

Source: `src/nscreen_graph/spark/sparksql/NScreenLrThrotleExReasonResult.sql` — a pure filter over `stage in ('initial','stage1','stage2','louvain')`, with **no per-UID deduplication** and no window function of any kind.

So every ambiguity that survived the preceding phases is still here: three rows for `v6`, two for `v13`, one for everyone else. In production the same is true at scale — one observed UID carries 16 matchids into the result.

(`matchid <> 0`, the `S_` stable prefix, and the junk-uid regexes are skipped — not applicable to the toy schema.)

One thing worth knowing that the toy cannot show, because the propagation stages are 1:1 by construction: in production the four result stages are **UID-disjoint** (measured: zero overlap). A UID is therefore never assigned by two different mechanisms, which means every duplicate row in the final result traces back to vendor ambiguity in `initial` — never to propagation and Louvain disagreeing about one person.

In [25]:
result = make_table(f"""
select uid, matchid, reason, stage
from {S}.nscreen_toy_staged
where stage in ('initial', 'stage1', 'stage2', 'louvain')
""", "nscreen_toy_result", recreate=True)

result.sort_values(["uid", "matchid"])

Dropping table iceberg.jteixeira_ipa.nscreen_toy_result
Query executed in 1.982 seconds
Creating table iceberg.jteixeira_ipa.nscreen_toy_result


,uid,matchid,reason,stage
7,v1,201,LRTHEX,initial
11,v10,202,LRTH,initial
12,v13,202,LRTH,initial
5,v13,203,LR,initial
6,v14,203,LR,initial
24,v15,201,LRTHEX,initial
26,v16,302,EX,initial
28,v17,201,IC,stage1
2,v18,201,IC,stage2
1,v19,901,LU,louvain


## Validation

The remaining cells assert the exact expected output of every step, so a clean run is proof that the narrative above matches what the SQL actually does. Any `FAIL` means the walkthrough is wrong, not the data.

### 1. Full staged table — 58 rows

Up from 39 in the earlier SQL-literal version of this notebook, for three reasons, all of them production-faithful:

- every LiveRamp-rooted row now appears **twice** (once in `lrth` as Job 1 left it, once in `initial` as Job 2 re-stamped it),
- the `L6`/`E3` group adds the `LREX` case, and
- the `v9`↔`v24`/`v25`/`v26` edges make four more already-resolved UIDs graph-connected, so `connected_ns` copies them too (still no new matchid assignment).

| stage | rows |
|---|---|
| `initial` | 24 |
| `lrth` | 21 |
| `connected_ns` | 8 |
| `stage1` | 2 |
| `stage2` | 1 |
| `louvain` | 2 |

In [26]:
expected_staged = [
    # --- initial: Job 2's output alone (every lrth row re-stamped, plus ex_out) ---
    ("initial", "v1", 201, "LRTHEX"),    # L1 -- gained EX from v3's evidence alone
    ("initial", "v2", 201, "LRTHEX"),    # L1
    ("initial", "v3", 201, "LRTHEX"),    # L1 -- the UID that supplied the evidence
    ("initial", "v4", 202, "LRTH"),      # L2
    ("initial", "v5", 109, "TH"),        # T9
    ("initial", "v6", 103, "TH"),        # T3
    ("initial", "v6", 104, "TH"),        # T4
    ("initial", "v6", 105, "TH"),        # T5
    ("initial", "v9", 201, "LRTHEX"),    # L1
    ("initial", "v10", 202, "LRTH"),     # L2
    ("initial", "v13", 202, "LRTH"),     # L2
    ("initial", "v13", 203, "LR"),       # L3
    ("initial", "v14", 203, "LR"),       # L3
    ("initial", "v15", 201, "LRTHEX"),   # L1 -- Experian-only, inherited via E1
    ("initial", "v16", 302, "EX"),       # E2 -- no alignment
    ("initial", "v22", 204, "LR"),       # L4 -- tie-rejected
    ("initial", "v23", 205, "LR"),       # L5 -- tie-rejected
    ("initial", "v24", 106, "TH"),       # T6
    ("initial", "v25", 106, "TH"),       # T6
    ("initial", "v26", 106, "TH"),       # T6
    ("initial", "v27", 106, "TH"),       # T6
    ("initial", "v30", 206, "LREX"),     # L6 -- production-only reason
    ("initial", "v31", 206, "LREX"),     # L6
    ("initial", "v32", 206, "LREX"),     # L6 -- Experian-only, inherited via E3

    # --- lrth: ALL of Job 1's output, pre-Experian ---
    ("lrth", "v1", 201, "LRTH"),         # L1 -- no EX yet
    ("lrth", "v2", 201, "LRTH"),         # L1
    ("lrth", "v3", 201, "LRTH"),         # L1
    ("lrth", "v4", 202, "LRTH"),         # L2
    ("lrth", "v5", 109, "TH"),           # T9
    ("lrth", "v6", 103, "TH"),           # T3
    ("lrth", "v6", 104, "TH"),           # T4
    ("lrth", "v6", 105, "TH"),           # T5
    ("lrth", "v9", 201, "LRTH"),         # L1
    ("lrth", "v10", 202, "LRTH"),        # L2
    ("lrth", "v13", 202, "LRTH"),        # L2
    ("lrth", "v13", 203, "LR"),          # L3
    ("lrth", "v14", 203, "LR"),          # L3
    ("lrth", "v22", 204, "LR"),          # L4
    ("lrth", "v23", 205, "LR"),          # L5
    ("lrth", "v24", 106, "TH"),          # T6
    ("lrth", "v25", 106, "TH"),          # T6
    ("lrth", "v26", 106, "TH"),          # T6
    ("lrth", "v27", 106, "TH"),          # T6
    ("lrth", "v30", 206, "LR"),          # L6 -- becomes LREX in initial
    ("lrth", "v31", 206, "LR"),          # L6

    # --- connected_ns: graph-connected copies, reason NA ---
    ("connected_ns", "v1", 201, "NA"),   # L1
    ("connected_ns", "v6", 103, "NA"),   # T3 -- ambiguous seed, all 3 copied
    ("connected_ns", "v6", 104, "NA"),   # T4
    ("connected_ns", "v6", 105, "NA"),   # T5
    ("connected_ns", "v9", 201, "NA"),   # L1 -- newly connected via v24/v25/v26
    ("connected_ns", "v24", 106, "NA"),  # T6 -- newly connected via v9
    ("connected_ns", "v25", 106, "NA"),  # T6
    ("connected_ns", "v26", 106, "NA"),  # T6

    # --- propagation and Louvain ---
    ("stage1", "v17", 201, "IC"),        # L1, one hop from v1
    ("stage1", "v20", 105, "IC"),        # T5 wins the T3/T4/T5 tie (largest matchid)
    ("stage2", "v18", 201, "IC"),        # L1, one hop from v17
    ("louvain", "v19", 901, "LU"),       # C1
    ("louvain", "v21", 901, "LU"),       # C1
]

staged_all = q(f"select stage, uid, matchid, reason from {S}.nscreen_toy_staged")
check("FULL STAGED TABLE (58 rows)", staged_all, expected_staged,
      ["stage", "uid", "matchid", "reason"])

PASS  FULL STAGED TABLE (58 rows) (58 rows)


True

In [27]:
staged_all


,stage,uid,matchid,reason
0,initial,v15,201,LRTHEX
1,initial,v32,206,LREX
2,initial,v16,302,EX
3,lrth,v3,201,LRTH
4,connected_ns,v24,106,NA
5,lrth,v27,106,TH
6,lrth,v6,105,TH
7,lrth,v26,106,TH
8,lrth,v24,106,TH
9,lrth,v5,109,TH


### 2. The diagnostic query — `distinct_stage` / `distinct_matchid` / `distinct_reason` per UID

The point of this query is to separate *lineage* from *ambiguity*:

- **`distinct_stage > 1` is normal.** Every LiveRamp-rooted UID has it, because Job 1 wrote it to `lrth` and Job 2 re-stamped it into `initial`. `v1` reaches 3 by also being copied into `connected_ns`.
- **`distinct_reason > 1` is usually harmless** — it means a row picked up an `EX` suffix (`v3`, `v30`) or an `NA` placeholder (`v1`) between stages.
- **`distinct_matchid > 1` is the real ambiguity signal**, and only `v6` (3) and `v13` (2) have it.

In [28]:
diag = q(f"""
select uid,
       count(distinct stage)   as distinct_stage,
       count(distinct matchid) as distinct_matchid,
       count(distinct reason)  as distinct_reason
from {S}.nscreen_toy_staged
group by uid
""")

expected_diag = [
    ("v1", 3, 1, 3),    # lrth(LRTH) -> initial(LRTHEX) -> connected_ns(NA)
    ("v2", 2, 1, 2),    # LRTH -> LRTHEX
    ("v3", 2, 1, 2),    # LRTH -> LRTHEX
    ("v4", 2, 1, 1),    # LRTH both stages
    ("v5", 2, 1, 1),    # TH both stages
    ("v6", 3, 3, 2),    # THROTLE-SIDE AMBIGUITY: 3 matchids x 3 stages
    ("v9", 3, 1, 3),    # LRTH -> LRTHEX -> NA, now also graph-connected via v24/v25/v26
    ("v10", 2, 1, 1),
    ("v13", 2, 2, 2),   # LIVERAMP-SIDE AMBIGUITY: L2 and L3
    ("v14", 2, 1, 1),
    ("v15", 1, 1, 1),   # ex_out only -> initial only
    ("v16", 1, 1, 1),   # ex_out only
    ("v17", 1, 1, 1),   # stage1 only
    ("v18", 1, 1, 1),   # stage2 only
    ("v19", 1, 1, 1),   # louvain only
    ("v20", 1, 1, 1),   # stage1 only
    ("v21", 1, 1, 1),   # louvain only
    ("v22", 2, 1, 1),   # looks clean -- nothing records its group's rejected merge
    ("v23", 2, 1, 1),
    ("v24", 3, 1, 2),   # lrth/initial (TH) -> connected_ns (NA), graph-connected via v9
    ("v25", 3, 1, 2),
    ("v26", 3, 1, 2),
    ("v27", 2, 1, 1),   # not wired to v9 -- stays plain lrth/initial lineage
    ("v30", 2, 1, 2),   # LR -> LREX
    ("v31", 2, 1, 2),   # LR -> LREX
    ("v32", 1, 1, 1),   # ex_out only, already LREX
]

check("DIAGNOSTIC QUERY", diag, expected_diag,
      ["uid", "distinct_stage", "distinct_matchid", "distinct_reason"])
diag.sort_values("uid")

PASS  DIAGNOSTIC QUERY (26 rows)


,uid,distinct_stage,distinct_matchid,distinct_reason
1,v1,3,1,3
14,v10,2,1,1
20,v13,2,2,2
11,v14,2,1,1
8,v15,1,1,1
17,v16,1,1,1
9,v17,1,1,1
18,v18,1,1,1
12,v19,1,1,1
3,v2,2,1,2


### 3. Ambiguity survives into the final result table

The final job filters `stage in ('initial','stage1','stage2','louvain')` with no per-UID dedup, so 29 rows come out: 24 from `initial`, plus 2 + 1 + 2 from propagation and Louvain. Only `v6` and `v13` carry more than one row, and both are vendor ambiguity — no UID is claimed by two different *mechanisms*.

In [29]:
res_counts = q(f"""
select uid, count(*) as n, count(distinct matchid) as dm
from {S}.nscreen_toy_result
group by uid
""").set_index("uid")

assert res_counts.loc["v6", "n"] == 3 and res_counts.loc["v6", "dm"] == 3, "v6 should keep 3 matchids"
assert res_counts.loc["v13", "n"] == 2 and res_counts.loc["v13", "dm"] == 2, "v13 should keep 2 matchids"
single = res_counts.drop(["v6", "v13"])
assert (single["n"] == 1).all(), f"unexpected multi-row uids: {single[single.n > 1].index.tolist()}"
print("PASS  final result: v6 keeps 3 matchids, v13 keeps 2, everyone else exactly 1")
res_counts.sort_index()

PASS  final result: v6 keeps 3 matchids, v13 keeps 2, everyone else exactly 1


,n,dm
uid,,
v1,1,1
v10,1,1
v13,2,2
v14,1,1
v15,1,1
v16,1,1
v17,1,1
v18,1,1
v19,1,1


### 4. Edge case answer: which LiveRamp id does T6 keep — L4, L5, or both?

**Both — but unmerged.** The tie (`(T6,L4)` cnt=1 vs `(T6,L5)` cnt=1) is rejected by the `count1 = 1` filter, so no mapping exists for T6. The one "person" behind T6 ends up under **three** matchids in the result: `L4` (v22), `L5` (v23), `T6` (v24…v27). Alignment preserved the fragmentation instead of resolving it.

In [30]:
t6_family = q(f"""
select uid, matchid, reason from {S}.nscreen_toy_result
where uid in ('v22','v23','v24','v25','v26','v27')
""")

check("T6/L4/L5 edge case in final result", t6_family, [
    ("v22", 204, "LR"), ("v23", 205, "LR"),        # L4, L5
    ("v24", 106, "TH"), ("v25", 106, "TH"),        # T6
    ("v26", 106, "TH"), ("v27", 106, "TH"),        # T6
], ["uid", "matchid", "reason"])

mids = sorted(set(t6_family["matchid"]))
assert mids == [106, 204, 205], mids   # T6, L4, L5
print("One Throtle person-group fractured into 3 matchids:", mids)
t6_family.sort_values("uid")

PASS  T6/L4/L5 edge case in final result (6 rows)
One Throtle person-group fractured into 3 matchids: [106, 204, 205]


,uid,matchid,reason
1,v22,204,LR
0,v23,205,LR
4,v24,106,TH
5,v25,106,TH
3,v26,106,TH
2,v27,106,TH


## Conclusions

If every cell above prints `PASS`, this replication is faithful to **production behaviour** (not merely to the SQL text) for:

- the mutual-best-rank mapping in both jobs, **including tie rejection** — `T6` overlapping `L4` and `L5` equally merges with *neither*, via the `count1 = 1` filter,
- **Job 1's real staging** — its `'initial'` literal is discarded, so all of its output sits in `lrth` and Job 2 reads the whole thing,
- **Experian aligning against LiveRamp-rooted groups**, producing `LREX` (`v30`, `v31`, `v32` via `L6`↔`E3`) — a reason that is unreachable if you trust the SQL's stage labels,
- **matchid-level merge propagation** — `v1`, `v2`, `v9` all become `LRTHEX` although only `v3` supplied the Experian evidence, exactly as the production UID `GvT54vuou0s4` behaves,
- group-level matchid inheritance — `v3` inheriting `L1` with no LiveRamp row of its own; `v15` inheriting `L1`, and `v32` inheriting `L6`, through Experian alignment,
- the `lr_out` pass-through vs `th_out` per-UID-existence asymmetry, and the group-vs-cookie granularity mismatch it creates,
- vendor ambiguity passing through untouched — `v6` ×3 and `v13` ×2, all the way into the result table,
- `connected_ns` copy semantics (row multiplication, `NA` reason) — including that being graph-connected is orthogonal to being resolved: the `v9`↔`v24`/`v25`/`v26` edges pull four already-assigned UIDs into `connected_ns` without producing a single new matchid assignment, because `filter_out_having_matchid` excludes any UID that already has one,
- propagation exclusivity and the `weight desc, matchid desc` tie-break (`v20` → `T5` = 105, decided **numerically** on bigint as in production),
- remainder selection (both endpoints unassigned + same geo) — Louvain itself simulated, not verified,
- **a geo-mismatched edge is excluded exactly like no edge at all** (`v33`↔`v34`) — same weight and assignment status as `v19`↔`v21`, dropped solely by `where g1.geo = g2.geo`, with zero trace left in the staged table.

**On matchid-cardinality inflation.** Alignment never *creates* a LiveRamp matchid and never merges one LiveRamp matchid into another — `lr_out` keeps `lr.matchid` unconditionally, so every LiveRamp id in the source survives to the result verbatim. The only lever alignment has is *reduction*: translating Throtle and Experian ids into LiveRamp's namespace. Every rejected merge leaves that reduction undone:

```text
result cardinality =   all LiveRamp matchids
                     + unmerged Throtle matchids
                     + unmerged Experian matchids
                     + Louvain communities
```

Fragmentation like `T6`/`L4`/`L5` — one person, three ids — is therefore *preserved*, not resolved. And because a losing LiveRamp group is never folded into a winning one, a person whose cookies span *n* LiveRamp groups ends up under at least *n* matchids no matter how strong the evidence.

### What this notebook still does NOT establish

- **The `count2` collateral rejection** — a Throtle group with exactly one candidate partner still losing its merge because a rival group ties over the same LiveRamp group. Needs two competing Throtle groups; verified in production on LiveRamp matchid `1672929395374000007`.
- **Job 2 tie rejection**, the only channel through which Experian evidence is discarded (verified in production on `lrth` matchid `-7674333472984662009`).
- **The ten-UID propagation cap** and the resulting UID loss — needs a matchid with more than ten contenders. In production 3.1M matchids saturate it per round, and because the cap is applied *after* each UID is reduced to one candidate, UIDs are dropped entirely while their discarded alternatives keep free slots.
- **Negative matchids.** Real murmur3 hashes are frequently negative, which makes the `matchid desc` tie-break arbitrary in practice; the toy's ascending values make `v20`→`T5` look principled when it is not.
- `householdid` / `stable` handling, including the stable weight-doubling in propagation, and the real `grouped` keying on `(uid, matchid, householdid, stable)`.
- Junk-uid filters, the `S_` stable prefix, and the real Louvain UDAF internals.

For all of the above with production measurements, see [`provider-alignment-phase.md`](docs/analysis/provider-alignment-phase.md) and [`provider-alignment-defects.md`](docs/analysis/provider-alignment-defects.md).